# MongoDB Schema Design Workshop
## Building a Loyalty Platform for KFC

Welcome! In this hands-on workshop you will design the data model for a
fast-food loyalty program step by step, using a real Atlas cluster.

**Agenda**

| Part | Topic |
|---|---|
| 0 | Setup & connect to Atlas |
| 1 | Embedding vs. Referencing — the foundational decision |
| 2 | The Extended Reference Pattern |
| 3 | Step-by-step schema design walkthrough |
| 4 | Schema patterns deep dive: Computed, Attribute, Outlier, Inheritance — and the Unbounded Array anti-pattern |
| 5 | Final consolidated schema + capstone exercise |
| 6 | Wrap-up & cheat sheet |

> 💡 Each pattern section follows the same flow: **Problem → Solution → Demo → Exercise**.
> Exercises have a collapsible `Solution` section below them — try it yourself first!

---
# Part 0 — Setup

Each attendee shares **one Atlas cluster** but works in **their own database**
on that cluster. Your `.env` file controls which database you use
(`DB_NAME`) — make sure it's unique to you. See `README.md` if you haven't
set this up yet.

In [1]:
import sys
sys.path.append("..")  # so we can import workshop_utils.py from the repo root

from workshop_utils import (
    get_db, reset_workshop_db, pprint_doc, pprint_docs,
    explain_summary, time_it, doc_size_bytes,
    generate_stores, generate_menu_items, generate_customers,
    generate_orders, generate_point_transactions_for_customer,
    generate_polymorphic_rewards,
)
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from bson import ObjectId

db = get_db()

Connected to Atlas cluster. Using database: 'kfc_loyalty_niko'


In [2]:
# Start from a clean slate. Safe to re-run at any point in the workshop.
reset_workshop_db(db)

Dropped collections in 'kfc_loyalty_niko': customers, stores, menu_items, orders, point_transactions, rewards, redemptions, customer_overflow


---
# Part 1 — Embedding vs. Referencing

This is the single most important decision in MongoDB schema design. Get it
right and your queries are fast and simple; get it wrong and you'll be
fighting the database forever.

### Decision framework

| Question | Favors Embedding | Favors Referencing |
|---|---|---|
| **Cardinality** | 1:1 or 1:few | 1:many or 1:squillions |
| **Read together?** | Yes, almost always fetched together | No, usually accessed independently |
| **Growth** | Bounded, stays small | Unbounded / grows over time |
| **Update frequency** | Rarely changes, or changes together | Child updates far more often than parent |
| **Atomicity needs** | Need single-document atomic update | Can tolerate multi-document consistency |
| **Document size** | Stays well under 16MB | Would risk approaching 16MB |

Rule of thumb: **embed what you query together and what stays small;
reference what grows without bound or is queried independently.**

### 1.1 Embedding example — Customer contact info (1:1, read together)

A customer's contact info is always read *with* the customer, rarely
changes, and is small and bounded. Perfect candidate for embedding.

In [3]:
sample_customer = {
    "_id": ObjectId(),
    "name": "Jordan Lee",
    "tier": "Gold",
    # Embedded: 1:1, always read together with the customer, small & stable
    "contactInfo": {
        "email": "jordan.lee@example.com",
        "phone": "+1-502-555-0199",
        "address": {"city": "Louisville", "state": "KY", "zip": "40202"},
    },
}
db.customers.insert_one(sample_customer)
pprint_doc(db.customers.find_one({"_id": sample_customer["_id"]}))

{
  "_id": "6ad9cfb78bbaa07eb0b0dc86",
  "name": "Jordan Lee",
  "tier": "Gold",
  "contactInfo": {
    "email": "jordan.lee@example.com",
    "phone": "+1-502-555-0199",
    "address": {
      "city": "Louisville",
      "state": "KY",
      "zip": "40202"
    }
  }
}


### 1.2 Referencing example — Customer → Orders (1:many, unbounded)

A customer can place thousands of orders over years. Orders are usually
queried independently (e.g. "show me order #1234"), and the relationship is
unbounded — a classic case for **referencing**, not embedding.

In [4]:
stores = generate_stores(n=6)
menu_items = generate_menu_items()
db.stores.insert_many(stores)
db.menu_items.insert_many(menu_items)

customers = generate_customers(n=60, store_ids=[s["_id"] for s in stores])
db.customers.insert_many(customers)

# Orders REFERENCE their customer by id rather than embedding full order
# history inside the customer document.
order_example = {
    "_id": ObjectId(),
    "customerId": customers[0]["_id"],   # <-- reference, not embed
    "storeId": stores[0]["_id"],
    "items": [{"sku": "ZNG-BURGER", "qty": 1, "unitPrice": 7.49}],
    "total": 7.49,
    "orderedAt": datetime.now(timezone.utc),
}
db.orders.insert_one(order_example)

print("Customer doc (small, stable):")
pprint_doc(db.customers.find_one({"_id": customers[0]["_id"]}))
print("\nOrder doc (references customer by id):")
pprint_doc(db.orders.find_one({"_id": order_example["_id"]}))

Customer doc (small, stable):
{
  "_id": "6ad9cfbc8bbaa07eb0b0dc87",
  "name": "Melinda Cameron",
  "email": "perezantonio@example.com",
  "phone": "(541)639-5376x7242",
  "homeStoreId": "STORE-1005",
  "joinedAt": "2025-09-03T10:39:46.696000",
  "tier": "Bronze",
  "loyaltyStats": {
    "lifetimePoints": 0,
    "currentBalance": 0,
    "totalOrders": 0,
    "lastComputedAt": "2025-09-03T10:39:46.696000"
  },
  "hasExtendedHistory": false
}

Order doc (references customer by id):
{
  "_id": "6ad9cfbc8bbaa07eb0b0dcc3",
  "customerId": "6ad9cfbc8bbaa07eb0b0dc87",
  "storeId": "STORE-1000",
  "items": [
    {
      "sku": "ZNG-BURGER",
      "qty": 1,
      "unitPrice": 7.49
    }
  ],
  "total": 7.49,
  "orderedAt": "2026-10-22T08:56:28.689000"
}


### 📝 Exercise 1

For each relationship below, decide **embed** or **reference**, and justify
using the decision framework table above:

1. `Customer` ↔ `StoreVisit` (a check-in event each time a customer visits a physical store — can happen hundreds of times/year)
2. `Order` ↔ `OrderItems` (the 1-6 line items on a single order)
3. `Customer` ↔ `FavoriteStore` (the one store a customer marks as their default pickup location)

Write your answers before expanding the solution below.

<details>
<summary>✅ Click to reveal solution</summary>

1. **`Customer` ↔ `StoreVisit` → Reference.** Unbounded 1:many (hundreds/year,
   growing forever), usually queried independently (e.g. "visits in the last
   30 days" for an analytics job), not read together with the customer
   profile on every request. Embedding would eventually create an
   unbounded array (see Part 4.1 anti-pattern!).

2. **`Order` ↔ `OrderItems` → Embed.** 1:few, always read and written
   together (you never fetch an order without its items), bounded in the
   common case (a handful of items). This is the textbook embedding case —
   *but* see Part 4.4 (Outlier Pattern) for what happens when a rare
   "catering order" has 200 items instead of 3.

3. **`Customer` ↔ `FavoriteStore` → Reference (as an id), not embed the full store doc.**
   It's 1:1, but the *store* itself is a shared, independently-updated
   entity referenced by many customers — embedding a full copy of the store
   document inside every customer would duplicate data and go stale when
   the store's hours/address change. Store a `favoriteStoreId` reference
   instead. (If you only need 1-2 display fields like the store's name,
   this is actually a great candidate for the **Extended Reference
   Pattern** — see Part 2!)

</details>

---
# Part 2 — The Extended Reference Pattern

### The problem

We just decided `Order → Customer` should be a **reference** (1:many,
unbounded). But our app's order-list screen needs to show the customer's
**name and loyalty tier** next to every order. With a pure reference, every
single order-list query needs a `$lookup` join back to `customers`.

At scale (thousands of orders listed per page load, across many
concurrent users), that's a lot of joins for data that barely ever changes.

### The solution

**Extended Reference Pattern**: keep the reference (`customerId`), but also
duplicate a *small, slow-changing* subset of the referenced document's
fields directly into the referencing document. You get reference semantics
(no duplication of the *whole* customer) plus embedding-like read
performance for the fields you actually display.

In [ ]:
# Rebuild `orders` using the Extended Reference Pattern: each order embeds
# a small snapshot (customerId, displayName, tier) instead of just customerId.
db.orders.drop()
store_ids = [s["_id"] for s in stores]
orders = generate_orders(customers, store_ids, menu_items, n=10000)
db.orders.insert_many(orders)

print("Order with extended reference:")
pprint_doc(db.orders.find_one())

### Side-by-side comparison

Let's prove the point: build a **pure-reference** sibling collection with
only `customerId`, then compare a "list recent orders with customer name
and tier" query using `$lookup` vs. using the extended reference fields.

A single comparison at a small page size (e.g. 20 rows) can be misleading:
`$lookup` on an indexed field is cheap *per row*, so for a handful of rows
the extra cost is tiny compared to normal network/measurement noise. The
real story is that the `$lookup` cost **scales with how many rows you
join** (think: an admin export, or a user scrolling through an infinite
feed), while the extended reference query's cost does not — so let's sweep
across page sizes and watch the gap widen.

In [ ]:
# Pure-reference version of the same data, for comparison only.
db.orders_pure_ref.drop()
pure_ref_docs = [
    {
        "_id": o["_id"],
        "customerId": o["customerRef"]["customerId"],
        "storeId": o["storeId"],
        "items": o["items"],
        "total": o["total"],
        "orderedAt": o["orderedAt"],
    }
    for o in orders
]
db.orders_pure_ref.insert_many(pure_ref_docs)

db.orders.create_index([("orderedAt", -1)])
db.orders_pure_ref.create_index([("orderedAt", -1)])
db.customers.create_index([("_id", 1)])  # default, shown for clarity
print("Collections ready for comparison.")

In [ ]:
import statistics

LIMITS = [100, 500, 2000, 8000]
N_TRIALS = 5


def lookup_pipeline(limit):
    return [
        {"$sort": {"orderedAt": -1}},
        {"$limit": limit},
        {"$lookup": {
            "from": "customers",
            "localField": "customerId",
            "foreignField": "_id",
            "as": "customer",
        }},
        {"$unwind": "$customer"},
        {"$project": {
            "total": 1, "orderedAt": 1,
            "displayName": "$customer.name", "tier": "$customer.tier",
        }},
    ]


def make_lookup_query(limit):
    pipeline = lookup_pipeline(limit)
    return lambda: list(db.orders_pure_ref.aggregate(pipeline))


def make_extended_query(limit):
    return lambda: list(
        db.orders.find({}, {"total": 1, "orderedAt": 1, "customerRef": 1})
        .sort("orderedAt", -1)
        .limit(limit)
    )


# Build both query shapes for every page size, then prime all of them before
# taking any measurements. This keeps first-use/plan/cache effects out of the
# first timed comparison rather than warming each strategy only on demand.
queries = {
    limit: (make_lookup_query(limit), make_extended_query(limit))
    for limit in LIMITS
}
for limit_index, limit in enumerate(LIMITS):
    lookup_query, extended_query = queries[limit]
    prime_order = (lookup_query, extended_query)
    if limit_index % 2:
        prime_order = prime_order[::-1]
    for query in prime_order:
        query()


# Time paired trials and alternate which strategy goes first to reduce
# systematic ordering bias. Five trials give a more stable median on Atlas.
timed_rows = {limit: {"lookup": [], "extended": []} for limit in LIMITS}
for trial in range(N_TRIALS):
    for limit_index, limit in enumerate(LIMITS):
        lookup_query, extended_query = queries[limit]
        run_lookup_first = (trial + limit_index) % 2 == 0
        trial_order = (
            [("lookup", lookup_query), ("extended", extended_query)]
            if run_lookup_first
            else [("extended", extended_query), ("lookup", lookup_query)]
        )
        for strategy, query in trial_order:
            timed_rows[limit][strategy].append(time_it(query)[1])


sweep_rows = []
for limit in LIMITS:
    lookup_ms = statistics.median(timed_rows[limit]["lookup"])
    extended_ms = statistics.median(timed_rows[limit]["extended"])
    sweep_rows.append({
        "limit (rows joined/returned)": limit,
        "Pure Reference $lookup (ms)": round(lookup_ms, 2),
        "Extended Reference (ms)": round(extended_ms, 2),
        "Speedup": round(lookup_ms / extended_ms, 1) if extended_ms > 0 else None,
    })

sweep_df = pd.DataFrame(sweep_rows)
sweep_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(sweep_df["limit (rows joined/returned)"], sweep_df["Pure Reference $lookup (ms)"],
        marker="o", label="Pure Reference ($lookup)", color="crimson")
ax.plot(sweep_df["limit (rows joined/returned)"], sweep_df["Extended Reference (ms)"],
        marker="o", label="Extended Reference (no $lookup)", color="seagreen")
ax.set_title("Query latency vs. rows returned: $lookup join vs. extended reference")
ax.set_xlabel("Rows returned (e.g. page size / export size)")
ax.set_ylabel("Median wall-clock time (ms, 5 trials)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Mechanism check: at the largest page size, confirm WHY the $lookup
# version is slower via .explain() — more documents examined, not just
# more wall-clock time.
biggest_limit = LIMITS[-1]
pipeline = lookup_pipeline(biggest_limit)

explain_ext = db.orders.find(
    {}, {"total": 1, "orderedAt": 1, "customerRef": 1}
).sort("orderedAt", -1).limit(biggest_limit).explain()

# Use db.command(..., explain=True) for aggregation explain (the
# collection.aggregate(explain=True) kwarg isn't supported by the driver,
# even in the latest pymongo — the driver explicitly raises a
# ConfigurationError pointing you to Database.command instead).
explain_lookup = db.command({
    "explain": {
        "aggregate": "orders_pure_ref",
        "pipeline": pipeline,
        "cursor": {},
    },
    "verbosity": "executionStats",
})

comparison_ext = explain_summary(explain_ext, label=f"Extended Reference (limit={biggest_limit})")

# Aggregation explain output has one entry per pipeline stage rather than a
# single top-level executionStats block, so we sum across stages: the
# initial $cursor stage (the collection scan feeding the pipeline) plus the
# $lookup stage (one extra doc examined per order, for the join). The exact
# $lookup strategy (indexed nested-loop vs. hash join) can vary, so we fall
# back gracefully if a field isn't reported for the chosen strategy.
cursor_stats = explain_lookup["stages"][0]["$cursor"]["executionStats"]
lookup_stage = next((s["$lookup"] for s in explain_lookup["stages"] if "$lookup" in s), {})
total_docs_examined = cursor_stats["totalDocsExamined"] + lookup_stage.get(
    "totalDocsExamined", cursor_stats["nReturned"]  # ~1 customer doc touched per order row
)
total_time_ms = cursor_stats["executionTimeMillis"] + sum(
    s.get("executionTimeMillisEstimate", 0) for s in explain_lookup["stages"]
)

comparison_lookup_row = pd.DataFrame([{
    "label": f"Pure Reference ($lookup, limit={biggest_limit})",
    "stage": "AGGREGATE + $lookup join",
    "executionTimeMillis": total_time_ms,
    "totalDocsExamined": total_docs_examined,
    "totalKeysExamined": cursor_stats["totalKeysExamined"] + lookup_stage.get("totalKeysExamined", 0),
    "nReturned": comparison_ext["nReturned"][0],
}])
pd.concat([comparison_lookup_row, comparison_ext], ignore_index=True)

Notice how the gap **widens as the page size grows**: at 100 rows the two
are close (the per-row $lookup cost is too small to matter yet), but by
4,000 rows the `$lookup` version is doing thousands of extra indexed
lookups against `customers` inside the aggregation pipeline, while the
extended reference query's cost barely changes — it never touches
`customers` at all, regardless of how many rows you ask for.

### ⚠️ The tradeoff: staleness

If a customer's `tier` changes (e.g. Silver → Gold), every **past** order
still shows the old tier snapshot — that's usually *fine* (orders are
historical records), but if you need the extended copy to stay fresh (e.g.
a "current tier badge" shown on an in-progress order), you must actively
keep it in sync:

- **App-level dual write**: when updating `customers.tier`, also update
  recent/open orders' `customerRef.tier` in the same request.
- **Change Streams**: watch `customers` for tier changes and asynchronously
  patch denormalized copies elsewhere.

Extended reference trades a small amount of write-side complexity for a
large amount of read-side simplicity and speed — worth it when reads vastly
outnumber writes, which is true for almost all loyalty-app order history UIs.

### 📝 Exercise 2

Orders also reference `storeId`. Design the **extended reference fields**
you'd add to the order document so a receipt screen can show the store's
name and city without a second query. What fields would you pick, and why
those and not others (e.g. not the store's full address or phone number)?

<details>
<summary>✅ Click to reveal solution</summary>

```json
{
  "_id": "...",
  "storeId": "STORE-1002",
  "storeRef": { "storeId": "STORE-1002", "name": "KFC Atlanta #1002", "city": "Atlanta, GA" },
  "...": "..."
}
```

Pick fields that are **(a) actually displayed on the order/receipt** and
**(b) rarely change** — `name` and `city` fit both criteria. Skip `address`
or `phone`: they're not needed on a receipt, so including them would just
bloat every order document for no read-performance benefit, and `address`
can actually change (store relocations) which would mean stale data stuck
on old receipts with no clear "this doesn't matter" justification like
there is for `tier`.

```python
db.orders.update_many({}, [
    {"$lookup": {
        "from": "stores", "localField": "storeId",
        "foreignField": "_id", "as": "_store"
    }},
    {"$set": {"storeRef": {
        "storeId": "$storeId",
        "name": {"$first": "$_store.name"},
        "city": {"$first": "$_store.city"},
    }}},
    {"$unset": "_store"},
])
```
</details>

---
# Part 3 — Step-by-Step Loyalty Schema Design Walkthrough

Good schema design starts from **access patterns**, not from entities. Let's
walk through the process the way you would on a real project.

### 3.1 Entities

- **Customer** — profile, tier, loyalty balance
- **Store** — physical location
- **Menu Item** — product catalog
- **Order** — a purchase, with line items
- **Point Transaction** — an earn or redeem event
- **Reward** — something redeemable with points (several types)
- **Redemption** — a reward actually claimed by a customer

### 3.2 Access patterns (write these down *before* modeling!)

| # | Access Pattern | Frequency | Latency need |
|---|---|---|---|
| 1 | Load customer profile + current points balance | Very high (every app open) | Fast |
| 2 | List a customer's recent orders with store/customer display info | High | Fast |
| 3 | Show a customer's full point transaction history (paginated) | Medium | Can be a bit slower |
| 4 | Record a new order + award points | High (every purchase) | Fast, must be reliable |
| 5 | Browse the reward catalog, filter by eligibility (tier, store, day) | High | Fast |
| 6 | Redeem a reward | Medium | Fast, must be reliable |
| 7 | Leaderboard — top spenders this month | Low (admin dashboard) | Can be slow/batched |

These access patterns are what drive every decision below — notice #1 wants
an **O(1) lookup** of a pre-aggregated balance (→ Computed Pattern), #3
implies an **unbounded, independently-paginated list** (→ reference, not
embed), and #5 implies **filtering on many optional criteria** (→ Attribute
Pattern).

### 3.3 Naive Schema v1 (don't do this!)

A natural first instinct is to embed everything a customer "has" directly
into the customer document, since it feels like 1:1 ownership:

```json
{
  "_id": "...",
  "name": "Jordan Lee",
  "tier": "Gold",
  "pointsHistory": [
    { "type": "earn", "points": 12, "timestamp": "2024-01-02T10:00:00Z" },
    { "type": "earn", "points": 8,  "timestamp": "2024-01-05T13:30:00Z" },
    { "type": "redeem", "points": -150, "timestamp": "2024-01-20T09:15:00Z" }
    // ... grows forever, once per purchase, for years
  ]
}
```

This *looks* reasonable for a brand-new customer with 3 transactions. Let's
find out what happens once that customer has been active for 3 years. 👇
(Part 4.1)

---
# Part 4 — Schema Patterns Deep Dive

## 4.1 Anti-Pattern: Unbounded Arrays

Let's actually build the naive schema from Part 3.3 and watch what happens
as the embedded array grows.

In [ ]:
db.customers_naive.drop()
naive_customer_id = ObjectId()
db.customers_naive.insert_one({
    "_id": naive_customer_id,
    "name": "Taylor Morgan",
    "tier": "Gold",
    "pointsHistory": [],
})

print("Starting doc size:",
      doc_size_bytes(db.customers_naive.find_one({"_id": naive_customer_id})),
      "bytes")

In [ ]:
# Simulate 3 years of weekly purchases (~150 transactions) pushed in
# batches, measuring how write latency and document size change as the
# EMBEDDED array grows.
batch_size = 15
n_batches = 20
sizes, batch_times = [], []

for b in range(n_batches):
    txns = generate_point_transactions_for_customer(naive_customer_id, n=batch_size)
    txn_subdocs = [{"type": t["type"], "points": t["points"],
                     "balanceAfter": t["balanceAfter"], "timestamp": t["timestamp"]}
                    for t in txns]

    _, elapsed_ms = time_it(
        db.customers_naive.update_one,
        {"_id": naive_customer_id},
        {"$push": {"pointsHistory": {"$each": txn_subdocs}}},
    )
    batch_times.append(elapsed_ms)
    sizes.append(doc_size_bytes(db.customers_naive.find_one({"_id": naive_customer_id})))

print(f"After {n_batches * batch_size} transactions embedded directly in the customer doc:")
print(f"  Final document size: {sizes[-1]:,} bytes")
print(f"  First batch write time: {batch_times[0]:.2f} ms | Last batch write time: {batch_times[-1]:.2f} ms")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
txn_counts = [(i + 1) * batch_size for i in range(n_batches)]

axes[0].plot(txn_counts, sizes, marker="o")
axes[0].set_title("Embedded document size vs. # transactions")
axes[0].set_xlabel("Total transactions embedded")
axes[0].set_ylabel("Document size (bytes)")

axes[1].plot(txn_counts, batch_times, marker="o", color="orange")
axes[1].set_title("Write latency per batch vs. # transactions")
axes[1].set_xlabel("Total transactions embedded")
axes[1].set_ylabel("Batch write time (ms)")

plt.tight_layout()
plt.show()

### Why this is a real problem, not just a theoretical one

1. **16MB document limit.** A typical transaction subdocument runs
   ~100-150 bytes once you include BSON field-name overhead. At ~130
   bytes/txn, you'd hit the 16MB ceiling around **120,000 transactions per
   customer** — a few per week for a *few hundred years* for a typical
   customer, but trivially reachable within months for a high-volume
   **outlier** account (think: a corporate catering account ordering daily
   for an office). See Part 4.4.
2. **Every write rewrites the whole array.** `$push` into a growing array
   means more bytes read, modified, and written back to disk (and
   replicated to every secondary) on every single purchase — even though
   the purchase only *added* a few hundred bytes of new data.
3. **You can't query a slice efficiently.** "Show me this customer's last 10
   transactions" means loading the *entire* document (however large) and
   slicing client-side — there's no way to index into the middle of an
   array efficiently at scale, and no way to paginate without re-fetching
   the whole array each time.
4. **Working-set bloat.** Even one customer with a huge array makes the
   `customers` collection's average document size balloon, hurting
   RAM cache efficiency for *every* customer, not just the outlier.

### The fix: move the unbounded relationship to its own collection

This is just the **Referencing** decision from Part 1, applied correctly:
`Customer : PointTransaction` is 1:many and unbounded → it belongs in its
own collection, referenced by `customerId`, not embedded.

In [ ]:
db.point_transactions.drop()
db.point_transactions.create_index([("customerId", 1), ("timestamp", -1)])

all_txns = []
for b in range(n_batches):
    all_txns.extend(generate_point_transactions_for_customer(naive_customer_id, n=batch_size))

ref_batch_times = []
for i in range(0, len(all_txns), batch_size):
    batch = all_txns[i:i + batch_size]
    _, elapsed_ms = time_it(db.point_transactions.insert_many, batch)
    ref_batch_times.append(elapsed_ms)

print(f"Referenced-collection batch write time: "
      f"first={ref_batch_times[0]:.2f}ms, last={ref_batch_times[-1]:.2f}ms "
      f"(stays flat regardless of history size)")
naive_doc_no_history = db.customers_naive.find_one({"_id": naive_customer_id}, {"pointsHistory": 0})
print(f"Customer doc stays tiny: {doc_size_bytes(naive_doc_no_history)} bytes "
      f"(excluding the old embedded array)")

> **Key takeaway:** if a child relationship is **unbounded** (grows without
> a natural, small cap), it almost always belongs in its own collection —
> regardless of how "owned" it conceptually feels by the parent.

---
## 4.2 Computed Pattern

### The problem

Access pattern #1 from Part 3 ("load customer profile + current points
balance") happens on *every app open* — it needs to be fast. But the
customer's current balance is really a derived value: the running sum of
all their point transactions. If we compute it live every time...

In [ ]:
# Seed a realistic dataset: customers already have orders (Part 2); now
# give each customer a substantial point-transaction history (2,000 each).
# With 60 customers this seeds about 120,000 transactions.
db.point_transactions.drop()
db.point_transactions.create_index([("customerId", 1), ("timestamp", -1)])

TRANSACTIONS_PER_CUSTOMER = 2000
for c in customers:
    txns = generate_point_transactions_for_customer(
        c["_id"], n=TRANSACTIONS_PER_CUSTOMER
    )
    db.point_transactions.insert_many(txns)

sample_customer_id = customers[0]["_id"]
print(
    f"Seeded {len(customers) * TRANSACTIONS_PER_CUSTOMER:,} point transactions "
    f"for {len(customers)} customers."
)

In [ ]:
# BEFORE: compute the customer's lifetime points from transaction history
# on every read. The same pipeline will be inspected with explain() below.
pipeline = [
    {"$match": {"customerId": sample_customer_id}},
    {"$group": {
        "_id": "$customerId",
        "lifetimePoints": {"$sum": {"$cond": [{"$gt": ["$points", 0]}, "$points", 0]}},
        "txnCount": {"$sum": 1},
    }},
]

live_result = list(db.point_transactions.aggregate(pipeline))
print("Live aggregation result:", live_result)

In [ ]:
# AFTER: apply the Computed Pattern. Maintain customers.loyaltyStats as a
# precomputed, incrementally-updated summary instead of recomputing it.

def record_point_transaction(db, customer_id, txn):
    """
    Writes a new transaction AND incrementally updates the precomputed
    summary on the customer doc, in one logical operation (production code
    should wrap this in a multi-document transaction for atomicity).

    Note: `currentBalance` is $set to the ledger's own `balanceAfter`
    (not re-derived by summing deltas) so it can never drift from the
    source-of-truth ledger, and naturally never goes negative since
    `balanceAfter` is floored at zero when the transaction is created.
    """
    db.point_transactions.insert_one(txn)
    db.customers.update_one(
        {"_id": customer_id},
        {
            "$inc": {"loyaltyStats.lifetimePoints": max(txn["points"], 0)},
            "$set": {
                "loyaltyStats.currentBalance": txn["balanceAfter"],
                "loyaltyStats.lastComputedAt": datetime.now(timezone.utc),
            },
        },
    )

# Backfill loyaltyStats once from existing history (bootstrap), then every
# NEW transaction updates it incrementally instead of recomputation.
for c in customers:
    agg = list(db.point_transactions.aggregate([
        {"$match": {"customerId": c["_id"]}},
        {"$group": {
            "_id": "$customerId",
            "lifetimePoints": {"$sum": {"$cond": [{"$gt": ["$points", 0]}, "$points", 0]}},
        }},
    ]))
    lifetime_points = agg[0]["lifetimePoints"] if agg else 0

    last_txn = db.point_transactions.find_one(
        {"customerId": c["_id"]}, sort=[("timestamp", -1)]
    )
    current_balance = last_txn["balanceAfter"] if last_txn else 0

    db.customers.update_one({"_id": c["_id"]}, {"$set": {
        "loyaltyStats.lifetimePoints": lifetime_points,
        "loyaltyStats.currentBalance": current_balance,
        "loyaltyStats.lastComputedAt": datetime.now(timezone.utc),
    }})

# Mechanism check: wall-clock time to a shared Atlas cluster includes
# network round-trip latency. Explain execution stats instead show how much
# work MongoDB performs for each strategy on the server.
explain_live = db.command({
    "explain": {
        "aggregate": "point_transactions",
        "pipeline": pipeline,
        "cursor": {},
    },
    "verbosity": "executionStats",
})

explain_computed = db.customers.find(
    {"_id": sample_customer_id}, {"loyaltyStats": 1}
).explain()

# MongoDB versions expose aggregation executionStats in different places:
# newer explain output may have top-level executionStats, while other versions
# put them under the initial $cursor stage in stages[]. Support both shapes.
aggregate_stats = explain_live.get("executionStats")
if aggregate_stats is not None:
    live_execution_ms = aggregate_stats.get("executionTimeMillis")
    live_docs_examined = aggregate_stats.get("totalDocsExamined")
    live_keys_examined = aggregate_stats.get("totalKeysExamined")
else:
    stages = explain_live.get("stages", [])
    cursor_stage = next(
        (stage["$cursor"] for stage in stages if "$cursor" in stage), {}
    )
    cursor_stats = cursor_stage.get("executionStats", {})
    live_execution_ms = cursor_stats.get("executionTimeMillis", 0) + sum(
        stage.get("executionTimeMillisEstimate", 0) for stage in stages
    )
    live_docs_examined = cursor_stats.get("totalDocsExamined")
    live_keys_examined = cursor_stats.get("totalKeysExamined")

comparison_live = pd.DataFrame([{
    "label": "Live aggregation ($match + $group)",
    "stage": "AGGREGATE",
    "executionTimeMillis": live_execution_ms,
    "totalDocsExamined": live_docs_examined,
    "totalKeysExamined": live_keys_examined,
    "nReturned": len(live_result),
}])
comparison_computed = explain_summary(
    explain_computed, label="Computed Pattern (indexed customer lookup)"
)
pd.concat([comparison_live, comparison_computed], ignore_index=True)

### The tradeoff

The Computed Pattern trades a tiny bit of write-side bookkeeping (`$inc` on
every transaction) for O(1) reads on the hottest access pattern in the app.
The risk is **drift**: if a transaction is ever inserted *without* going
through `record_point_transaction`, the summary silently goes stale. In
production, pair this with either:
- a **multi-document transaction** wrapping both writes, or
- a **periodic reconciliation job** that recomputes and corrects drift.

### 📝 Exercise 3

Apply the Computed Pattern to access pattern #4 (recording a new order).
Add `loyaltyStats.totalOrders` and `loyaltyStats.totalSpend` fields that are
incrementally updated every time an order is placed, instead of being
computed by aggregating the `orders` collection.

<details>
<summary>✅ Click to reveal solution</summary>

```python
def record_order(db, order_doc):
    db.orders.insert_one(order_doc)
    db.customers.update_one(
        {"_id": order_doc["customerRef"]["customerId"]},
        {
            "$inc": {
                "loyaltyStats.totalOrders": 1,
                "loyaltyStats.totalSpend": order_doc["total"],
            },
            "$set": {"loyaltyStats.lastComputedAt": datetime.now(timezone.utc)},
        },
    )
    # The points-earning side of the order is still recorded through the
    # single source of truth for the ledger, record_point_transaction(),
    # so currentBalance/lifetimePoints never drift out of sync with it.
```

Same idea as `record_point_transaction`: the write path does a little more
work (`$inc` three or four counters) so that the read path for the
dashboard/profile screen stays a single indexed document lookup, no
aggregation required.
</details>

---
## 4.3 Attribute Pattern

### The problem

Access pattern #5: "browse the reward catalog, filter by eligibility (tier,
store, day of week, channel...)". Rewards have wildly different eligibility
criteria, and new criteria types get added over time by marketing. A naive
design puts every possible criterion as its own top-level field:

```json
{
  "_id": "RWD-1",
  "name": "Free Delivery",
  "minTier": "Silver",
  "minOrderValue": null,
  "validDayOfWeek": null,
  "eligibleStoreId": null,
  "channelRestriction": null
}
```

Most fields are `null` on most documents (sparse), and to query efficiently
you'd need a **separate index per field** (`minTier_1`, `minOrderValue_1`,
`validDayOfWeek_1`, ...) — expensive to maintain, and you're still stuck
rewriting the schema every time marketing invents a new criterion type.

### The solution

**Attribute Pattern**: move these similar, sparse, "same shape but
different name" fields into a `k`/`v` array. One **generic compound index**
on `(attributes.k, attributes.v)` covers queries against *any* criterion,
present or future.

In [ ]:
db.rewards.drop()

reward_eligibility_docs = [
    {"_id": "RWD-1", "name": "Free Delivery", "pointsCost": 75,
     "eligibility": [{"k": "minTier", "v": "Silver"}]},
    {"_id": "RWD-2", "name": "20% Off Order", "pointsCost": 200,
     "eligibility": [{"k": "minTier", "v": "Gold"}, {"k": "minOrderValue", "v": 15}]},
    {"_id": "RWD-3", "name": "Friday Zinger Deal", "pointsCost": 120,
     "eligibility": [{"k": "validDayOfWeek", "v": "Friday"}]},
    {"_id": "RWD-4", "name": "Drive-Thru Only Combo", "pointsCost": 100,
     "eligibility": [{"k": "channel", "v": "drive_thru"}]},
    {"_id": "RWD-5", "name": "Platinum VIP Bucket", "pointsCost": 300,
     "eligibility": [{"k": "minTier", "v": "Platinum"}]},
    {"_id": "RWD-6", "name": "Louisville Flagship Exclusive", "pointsCost": 90,
     "eligibility": [{"k": "eligibleStoreId", "v": "STORE-1000"}]},
]
db.rewards.insert_many(reward_eligibility_docs)

# ONE generic index covers every current and future eligibility criterion.
db.rewards.create_index([("eligibility.k", 1), ("eligibility.v", 1)])
print("Attribute-pattern index created.")

In [ ]:
# Query: rewards eligible for a Gold-tier customer
explain_gold = db.rewards.find(
    {"eligibility": {"$elemMatch": {"k": "minTier", "v": "Gold"}}}
).explain()
print("Gold-tier eligible rewards:")
pprint_docs(db.rewards.find({"eligibility": {"$elemMatch": {"k": "minTier", "v": "Gold"}}}))

# Query: rewards eligible on Fridays -- SAME index, different k/v, no schema change
explain_friday = db.rewards.find(
    {"eligibility": {"$elemMatch": {"k": "validDayOfWeek", "v": "Friday"}}}
).explain()

comparison = pd.concat([
    explain_summary(explain_gold, label="minTier=Gold"),
    explain_summary(explain_friday, label="validDayOfWeek=Friday"),
], ignore_index=True)
comparison

Both queries hit an `IXSCAN` on the **same single index**, despite filtering
on completely different criteria. Compare that to the naive top-level-field
design, which would need a brand new index (and a schema migration) every
time marketing dreams up a new eligibility rule.

---
## 4.4 Outlier Pattern

### The problem

Access pattern #2 says orders embed their line items (Part 1 told us 1-6
items is a perfect embedding case). That's true for **99% of orders** — but
KFC also fulfills **catering orders** for offices and events, which can
have **100+ line items**. If we let those embed normally:

In [ ]:
# A catering order embedded "normally" -- technically legal, but look at the
# size difference vs. a typical order.
catering_items = [
    {"sku": mi["_id"], "name": mi["name"], "qty": 20, "unitPrice": mi["price"]}
    for mi in (menu_items * 15)[:140]
]
catering_order_naive = {
    "_id": ObjectId(),
    "customerRef": {"customerId": customers[0]["_id"], "displayName": customers[0]["name"], "tier": customers[0]["tier"]},
    "storeId": stores[0]["_id"],
    "items": catering_items,
    "total": round(sum(i["qty"] * i["unitPrice"] for i in catering_items), 2),
    "orderedAt": datetime.now(timezone.utc),
}

typical_order = db.orders.find_one()
print(f"Typical order size:  {doc_size_bytes(typical_order):,} bytes ({len(typical_order['items'])} items)")
print(f"Catering order size: {doc_size_bytes(catering_order_naive):,} bytes ({len(catering_order_naive['items'])} items)")

No single document breaks the 16MB limit here — but mixing a handful of
these into the `orders` collection drags up the **average document size**,
hurting RAM working-set efficiency for the entire collection (most reads —
e.g. "show recent orders" list views — only need the item *count* and
*total*, not every line item, for 99% of orders). A few outliers shouldn't
force every reader to pay for data they don't need.

### The solution

**Outlier Pattern**: design for the common case (small, embedded items
array), detect when a document becomes an outlier, flip a flag, and move
the *overflow* portion into a separate collection that only outlier-aware
code paths need to query.

In [ ]:
db.order_items_overflow.drop()
ITEM_THRESHOLD = 20

def save_order_with_overflow(db, order_doc):
    items = order_doc["items"]
    if len(items) <= ITEM_THRESHOLD:
        order_doc["hasMoreItems"] = False
        order_doc["itemCount"] = len(items)
        db.orders.insert_one(order_doc)
        return

    # Outlier: keep a preview inline, flag it, overflow the rest.
    order_doc["itemCount"] = len(items)
    order_doc["hasMoreItems"] = True
    order_doc["items"] = items[:ITEM_THRESHOLD]
    db.orders.insert_one(order_doc)
    db.order_items_overflow.insert_one({
        "orderId": order_doc["_id"],
        "items": items[ITEM_THRESHOLD:],
    })

def get_order_items(db, order_id):
    """App-level read path: only queries the overflow collection for the
    rare outlier orders that actually need it."""
    order = db.orders.find_one({"_id": order_id})
    items = order["items"]
    if order.get("hasMoreItems"):
        overflow = db.order_items_overflow.find_one({"orderId": order_id})
        items = items + overflow["items"]
    return items

save_order_with_overflow(db, catering_order_naive)

stored_order = db.orders.find_one({"_id": catering_order_naive["_id"]})
print("Stored outlier order sample:")
pprint_doc(stored_order)
print(f"Stored order doc size: {doc_size_bytes(stored_order):,} bytes "
      f"(inline items: {len(stored_order['items'])}, itemCount flag: {stored_order['itemCount']})")

full_items = get_order_items(db, catering_order_naive["_id"])
print(f"get_order_items() reconstructs all {len(full_items)} items transparently when needed.")

> **Key takeaway:** design your schema for the *typical* document, not the
> exceptional one. When a small number of documents are genuine outliers,
> handle them explicitly (flag + overflow collection) rather than letting
> them dictate the shape of every document in the collection.

---
## 4.5 Inheritance (Polymorphic) Pattern

### The problem

Our `rewards` collection so far only has `eligibility` criteria (Part 4.3).
But rewards also come in genuinely different **types**, each with its own
type-specific fields: a free-item reward needs a `menuItemSku`, a
percent-discount reward needs a `percentage`, a delivery-waiver reward needs
a `waivedFeeAmount`, and a BOGO reward needs `eligibleSkus`. These types
share a common "base" shape (name, pointsCost, validity window) but diverge
below that.

### The solution

**Inheritance (Polymorphic) Pattern**: one collection, one `type`
discriminator field, shared base fields at the top level, and a
type-specific nested object. Enforce the per-type shape with `$jsonSchema`
validation using `oneOf`.

In [ ]:
db.rewards.drop()

reward_validator = {
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["_id", "type", "name", "pointsCost", "validFrom", "validTo"],
        "properties": {
            "type": {"enum": ["free_item", "percent_discount", "delivery_waiver", "bogo"]},
            "name": {"bsonType": "string"},
            "pointsCost": {"bsonType": "int", "minimum": 1},
        },
        "oneOf": [
            {"properties": {"type": {"enum": ["free_item"]}},
             "required": ["freeItem"]},
            {"properties": {"type": {"enum": ["percent_discount"]}},
             "required": ["percentDiscount"]},
            {"properties": {"type": {"enum": ["delivery_waiver"]}},
             "required": ["deliveryWaiver"]},
            {"properties": {"type": {"enum": ["bogo"]}},
             "required": ["bogo"]},
        ],
    }
}

db.create_collection("rewards", validator=reward_validator)
db.rewards.create_index([("type", 1)])

polymorphic_rewards = generate_polymorphic_rewards()
db.rewards.insert_many(polymorphic_rewards)
print(f"Inserted {len(polymorphic_rewards)} polymorphic reward documents.")
pprint_docs(db.rewards.find(), n=2)

In [ ]:
# Cross-type query: the reward catalog browse screen (access pattern #5)
# just wants "all currently redeemable rewards", regardless of type.
print("All rewards (any type):")
pprint_docs(db.rewards.find({}, {"name": 1, "type": 1, "pointsCost": 1}))

# Type-specific query: redemption flow for a specific reward type knows
# exactly which nested object to read.
print("\nOnly percent-discount rewards:")
pprint_docs(db.rewards.find({"type": "percent_discount"}))

In [ ]:
# Schema validation enforces that each type carries its required nested object.
from pymongo.errors import WriteError

malformed_reward = {
    "_id": "RWD-BAD",
    "type": "free_item",
    "name": "Broken Reward",
    "pointsCost": 50,
    "validFrom": datetime.now(timezone.utc),
    "validTo": datetime.now(timezone.utc),
    # Missing the required "freeItem" nested object for this type!
}

try:
    db.rewards.insert_one(malformed_reward)
except WriteError as e:
    print("Insert rejected by $jsonSchema validator, as expected:")
    print(f"  {e.details.get('errmsg', str(e))}")

Notice how this composes with the Attribute Pattern from 4.3 — in a real
system, every reward document would carry **both** a `type`-discriminated
shape (Inheritance Pattern, for the core redemption logic) **and** an
`eligibility` array (Attribute Pattern, for the flexible catalog-filtering
logic). Patterns aren't mutually exclusive; production schemas typically
layer several of them on the same collection.

---
# Part 5 — Final Consolidated Schema

### Pattern recap

| Pattern | Problem it solved | Where we applied it |
|---|---|---|
| **Referencing** | Unbounded 1:many relationships | `Customer → Orders`, `Customer → PointTransactions` |
| **Extended Reference** | Avoid `$lookup` joins for frequently-displayed, slow-changing fields | `Order.customerRef` (name, tier), `Order.storeRef` (name, city) |
| **Unbounded Array (anti-pattern)** | What happens if you embed an ever-growing child collection | `Customer.pointsHistory[]` — fixed by extracting to `point_transactions` |
| **Computed** | Avoid expensive live aggregation on hot read paths | `Customer.loyaltyStats` (lifetimePoints, currentBalance, totalOrders) |
| **Attribute** | Many sparse, similarly-shaped optional fields that need generic querying | `Reward.eligibility[]` (minTier, minOrderValue, validDayOfWeek, ...) |
| **Outlier** | A few documents need disproportionately more data than the typical document | `Order.items[]` + `order_items_overflow` for catering orders |
| **Inheritance (Polymorphic)** | Multiple related sub-types sharing a base shape, diverging below it | `Reward.type` discriminator (free_item / percent_discount / delivery_waiver / bogo) |

### Final collections

| Collection | Purpose |
|---|---|
| `customers` | Profile, tier, precomputed `loyaltyStats` |
| `stores` | Physical locations |
| `menu_items` | Product catalog |
| `orders` | Purchases, with extended-reference customer/store snapshots and outlier-aware items |
| `order_items_overflow` | Overflow line items for outlier (catering) orders |
| `point_transactions` | Append-only earn/redeem ledger, referenced by `customerId` |
| `rewards` | Polymorphic reward catalog with eligibility attributes |
| `redemptions` | A customer claiming a reward (references `customerId` + `rewardId`) |

### 📝 Capstone Exercise: Referral Program

KFC wants to add a **"refer a friend"** feature: an existing customer
shares a referral code; when a new customer signs up and places their first
order, both the referrer and referee get bonus points.

Design the schema changes needed to support this feature. Specifically:

1. Where does the referral relationship live — a new field, a new
   collection, or both?
2. Which pattern(s) from today would you apply, and why?
3. How would you efficiently answer: "how many successful referrals has
   customer X made, and how many bonus points have they earned from
   referrals?"

Sketch your answer (markdown/JSON is fine, code is a bonus) before checking
the solution.

**The full written solution is in [`CAPSTONE_SOLUTION.md`](../CAPSTONE_SOLUTION.md)
in the repo root** — try the exercise yourself first!

In [ ]:
# ✍️ Your capstone design goes here (markdown sketch, sample documents,
# and/or code). Compare against CAPSTONE_SOLUTION.md when you're done.

---
# Part 6 — Wrap-up

### Decision checklist

**Embed vs. Reference**
- [ ] Is the relationship 1:1 or 1:few, bounded, and always read together? → **Embed**
- [ ] Is it 1:many/unbounded, or queried independently of the parent? → **Reference**
- [ ] Do you need a reference's display fields on every read, and they rarely change? → **Extended Reference**

**Reach for a pattern when you recognize...**
- [ ] A hot read path needs a value you'd otherwise compute live every time → **Computed Pattern**
- [ ] Many optional, similarly-shaped fields that need flexible/generic querying → **Attribute Pattern**
- [ ] A small number of documents need disproportionately more data than the typical document → **Outlier Pattern**
- [ ] Several sub-types share a base shape but diverge in type-specific fields → **Inheritance (Polymorphic) Pattern**

**Red flag to avoid**
- [ ] An array inside a document that grows once per event, forever, with no natural cap → **Unbounded Array anti-pattern** — move it to its own referenced collection.

### What we covered today

1. The embedding vs. referencing decision framework, and the Extended
   Reference Pattern for solving the "I need a join but don't want to pay
   for one" problem.
2. Driving schema design from concrete access patterns, not just entity
   relationships.
3. Four production-grade schema patterns (Computed, Attribute, Outlier,
   Inheritance) and one classic anti-pattern (Unbounded Arrays), all
   demonstrated on a single coherent loyalty-platform schema.

Thanks for attending! 🍗